라이브러리 로드

In [ ]:
# Import or install Sionna
import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import drjit as dr
import mitsuba as mi
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import tensorflow as tf
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import plotly.graph_objects as go

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
    
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, ITURadioMaterial, \
    SceneObject, Camera, PathSolver, InteractionType, RadioMapSolver
from sionna.rt.utils import r_hat

import importlib
import js_utils as js
js = importlib.reload(js)

no_preview = False # Toggle to False to use the preview widget

1. Scene XML 파일 로드 및 재질 확인

In [ ]:
# XML 파일 경로 보정 및 로드
xml_path = "/home/js/js/sionna-rt/src/sionna/rt/scenes/HSV/HSV_flat.xml"
scene_dir = os.path.dirname(xml_path)

scene = load_scene(xml_path, merge_shapes=False)
scene.frequency = 28e9
# 재질(Material) 정보 확인
js.print_scene_materials(scene)

2. 재질 변경

In [ ]:
# 도로
red_road_mat = ITURadioMaterial(name="red_road_mat_19",
                                itu_type="concrete",
                                thickness=0.2,
                                color=[1,0,0])
scene.add(red_road_mat)

road_object_id = "elm__19"

if road_object_id in scene.objects:
    scene.objects[road_object_id].radio_material = red_road_mat
    print(f"[설정] ({road_object_id}) 색상을 변경했습니다.")

# 초록색 나무(wood) 재질 생성
green_wood_mat = ITURadioMaterial(name="green_wood_material",
                                  itu_type="wood",       # 요청하신 wood 타입
                                  thickness=0.2,
                                  color=[0.2, 0.6, 0.2]) # 초록색 (R, G, B)
scene.add(green_wood_mat)

# 변경할 오브젝트 아이디 리스트
wood_object_ids = [
    "map_2_osm_forest-itu_wood",
    "map_2_osm_forest-itu_wood-001",
    "map_2_osm_forest-itu_wood-002"
]

# 일괄 적용
for obj_id in wood_object_ids:
    if obj_id in scene.objects:
        scene.objects[obj_id].radio_material = green_wood_mat
        print(f"[설정 완료] ({obj_id})를 초록색 나무 재질로 변경했습니다.")
    else:
        print(f"[경고] ({obj_id})를 씬(scene)에서 찾을 수 없습니다.")

3. 도로 정점 추출

In [ ]:
road_object_id = "elm__19"
road_positions = js.get_road_positions_from_object(
    scene,
    road_object_id,
    dtype=np.float16,       # 한글 주석: 연산용 기본 타입
    round_decimals=3,       # 한글 주석: 화면 표시 자릿수만 줄임
)

if len(road_positions) == 0:
    print("[안내] 추출된 좌표가 없습니다.")
else:
    print("dtype:", road_positions.dtype)
    print("first road position:", road_positions[0])

4. 초기 위치 테스트

In [ ]:
# 설정 및 초기화
target_pos = np.array(road_positions[0], dtype=np.float32)
print("="*60 + f"\n[설정] 테스트 목표: {target_pos}\n" + "="*60)

# 장치 배치
scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

tx_positions = [[0, -264, 11]]
tx_names = ["Tx_1"]

# 기존 객체 제거
for name in tx_names + ["Car_Marker"]:
    if name in scene.transmitters:
        scene.remove(name)
if "rx_car" in scene.receivers:
    scene.remove("rx_car")
if "rx_vehicle" in scene.objects:
    scene.edit(remove="rx_vehicle")

# Tx 배치
for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=43)
    tx.transmit_antenna = scene.tx_array
    tx.look_at(target_pos)
    scene.add(tx)
    print(f" -> {tx_names[i]} 배치 완료.")

In [ ]:
# 자동차 Mesh 생성
car_material = ITURadioMaterial(
    name="car_material_rt",
    itu_type="metal",
    thickness=0.01,
    color=(0,1,1),
)
scene.add(car_material)

vehicle = SceneObject(
    fname=sionna.rt.scene.low_poly_car,
    name="rx_vehicle",
    radio_material=car_material,
)
scene.edit(add=[vehicle])

vehicle.scaling = mi.Vector3f(2.5, 2.5, 2.5)
vehicle.position = mi.Vector3f(float(target_pos[0]), float(target_pos[1]), 0.3)
vehicle.orientation = mi.Point3f(0.0, 0.0, np.pi/2)

# rx 추가
rx = Receiver(
    name="rx_car",
    position=np.array([target_pos[0], target_pos[1], 3.0], dtype=np.float32)
)
rx.receive_antenna = scene.rx_array
scene.add(rx)
print(" -> 자동차(Rx) 및 Mesh 배치 완료.")

# 시뮬레이션
print("[연산] 경로 계산 시작...")
solver = PathSolver()
paths = solver(scene, max_depth=5, samples_per_src=1000000, diffuse_reflection=True, diffraction=True)

# 시각화
cam = Camera(position=target_pos + np.array([0, 100, 100]), look_at=target_pos)
print("[시각화] 3D 뷰어 실행")
scene.preview(paths=paths, show_devices=True)

***메인 시뮬레이션***

경로 준비

In [ ]:
# 한글 주석: 먼저 XY에서 정점 인덱스 분포를 확인
js.plot_vertices_index(
    road_positions,
    width=1000,
    height=800,
    marker_size=5,
)

In [ ]:
# 한글 주석: 위 Cell 에서 본 인덱스를 직접 입력
if 'road_positions' not in globals():
    raise ValueError("❌ 'road_positions' 데이터가 없습니다.")

path_indices = [4, 16]

path_data = js.prepare_vehicle_path(
    road_positions=road_positions,
    path_indices=path_indices,
    z_offset=3,
    speed_ms=100.0 / 3.6,
    delta_t=0.5,
)

waypoints = path_data["waypoints"]
time_steps = path_data["time_steps"]
print("waypoints:", waypoints.shape, "| time_steps:", len(time_steps))
print(f"총 이동 거리: {path_data['total_distance']:.2f} m")
print(f"총 소요 시간: {path_data['total_time']:.2f} s")

In [ ]:
js.plot_vertices_xy(
    road_positions=road_positions,
    waypoints=waypoints,
    start_point=path_indices[0],
    end_point=path_indices[-1],
    figsize=(9, 7),
)

Scene 초기화 (Tx/Rx + Solver)

In [ ]:
tx_names, rx, solver, vehicle = js.setup_scene_for_vehicle_path(
    scene=scene,
    path_data=path_data,
    tx_positions=tx_positions,
    tx_names=tx_names,
    tx_power_dbm=43.0,
    display_radius=2.0,
    add_vehicle=True,
    vehicle_name="rx_vehicle",
    vehicle_scale=(2.5, 2.5, 2.5),
    vehicle_z=0.3,
    rx_z=3.0,
)

Interactive 위젯 실행

In [ ]:
# path_loss.ipynb - Interactive 위젯 셀(기존 cell 18) 교체 예시
slider, output_widget = js.create_vehicle_simulation_widgets(
    scene=scene,
    path_data=path_data,
    tx_names=tx_names,
    rx=rx,
    solver=solver,
    vehicle=vehicle,
    vehicle_z=0.3,
    rx_z=3.0,
    max_depth=3,
    max_num_paths_per_src=100000,
    samples_per_src=100000,
    diffuse_reflection=True,
    diffraction=True,
    synthetic_array=True,
    resolution=(800, 600),
)

print("▼ 슬라이더를 움직여보세요.")
display(slider, output_widget)

CIR 측정

In [ ]:
# 1. 이전 셀 등에서 미리 만들어둔 경로 데이터를 불러옵니다.
# path_data = prepare_vehicle_path(road_positions, speed_ms=60/3.6, use_centerline=True)

# 2. path_data 안에서 시간축 배열(time_steps)을 꺼내옵니다.
time_steps = path_data["time_steps"]

frame_idx = 11  # 보고 싶은 프레임 인덱스 (원하는 값으로 바꿔)
t = time_steps[frame_idx]

# 3. get_pos_at_time 대신 좀 전에 만든 get_state_at_time을 사용하여 위치와 속도를 동시에 가져옵니다.
current_pos, current_vel = js.get_state_at_time(path_data, t)

# 4. 위치/방향 업데이트 (속도까지 업데이트 해주는 것이 도플러 효과에 좋습니다!)
rx.position = current_pos
rx.velocity = current_vel

for name in tx_names:
    scene.transmitters[name].look_at(current_pos)

# 5. 경로 계산
paths = solver(scene, max_depth=3, samples_per_src=100000,
               diffuse_reflection=True, diffraction=True, synthetic_array=True)

# 6. CIR 추출
a, tau = paths.cir(out_type="tf", normalize_delays=False)

print("t =", t)
print("a shape:", a.shape)
print("tau shape:", tau.shape)

CIR 시각화

In [ ]:
# 한글 주석: 텐서/배열 모두 처리하기 위해 numpy로 변환
a_np = a.numpy() if hasattr(a, "numpy") else np.asarray(a)
tau_np = tau.numpy() if hasattr(tau, "numpy") else np.asarray(tau)

print("a shape:", a_np.shape)
print("tau shape:", tau_np.shape)

# 한글 주석: a에서 path 축 길이 확인 (기본 가정: 마지막에서 2번째 축이 path)
if a_np.ndim < 2:
    raise ValueError(f"a 차원이 예상보다 작습니다: ndim={a_np.ndim}, shape={a_np.shape}")

num_paths = a_np.shape[-2]
print("num_paths:", num_paths)

if num_paths == 0:
    print("⚠️ 현재 프레임에서 유효 경로가 없어 CIR를 그릴 수 없습니다.")
else:
    # 한글 주석: a는 [rx, rx_ant, tx, tx_ant, path, time] 형태를 우선 가정
    # time 축은 첫 번째(0)만 사용
    if a_np.ndim == 6:
        a_abs = np.abs(a_np[0, 0, 0, 0, :, 0])
    elif a_np.ndim == 5:
        a_abs = np.abs(a_np[0, 0, 0, 0, :])
    else:
        # 한글 주석: 예상 외 shape는 마지막 path 축 기준으로 평탄화
        a_abs = np.abs(a_np.reshape(-1, a_np.shape[-2])[:, :]).mean(axis=0)

    # 한글 주석: tau shape를 자동 처리
    # 흔한 경우 1) [rx, tx, path]  2) [rx, rx_ant, tx, tx_ant, path]
    if tau_np.ndim == 3:
        t_ns = tau_np[0, 0, :] / 1e-9
    elif tau_np.ndim == 5:
        t_ns = tau_np[0, 0, 0, 0, :] / 1e-9
    else:
        # 한글 주석: 예상 외 shape는 마지막 축(path) 기준으로 처리
        t_ns = tau_np.reshape(-1, tau_np.shape[-1])[0] / 1e-9

    # 한글 주석: 길이 불일치 방어
    n = min(len(t_ns), len(a_abs))
    t_ns = t_ns[:n]
    a_abs = a_abs[:n]

    if n == 0:
        print("⚠️ tau 또는 a_abs가 비어 있어 CIR를 그릴 수 없습니다.")
    else:
        plt.figure(figsize=(8, 4))
        plt.title("Channel Impulse Response")
        plt.stem(t_ns, a_abs, basefmt=" ")
        plt.xlim(left=0)
        plt.xlabel(r"$\tau$ [ns]")
        plt.ylabel(r"$|a|$")
        plt.grid(True, linestyle="--", alpha=0.6)
        plt.show()


CFR 시각화

In [ ]:
# 비-OFDM 단일 주파수 CFR
freqs = np.array([28e9], dtype=np.float32)  # 한 점만 보려면
h_freq = paths.cfr(
    frequencies=freqs,
    normalize=True,
    normalize_delays=True,
    out_type="numpy"
)
print("shape:", h_freq.shape)  # [..., num_time_steps, num_frequencies]
print("abs(h):", np.abs(h_freq)[0,0,0,0,0,0])

In [ ]:
# Cell 2: 비-OFDM 주파수 스윕 CFR

freqs = np.linspace(27.5e9, 28.5e9, 801, dtype=np.float32)  # 임의 스윕
h_freq = paths.cfr(
    frequencies=freqs,
    normalize=True,
    normalize_delays=True,
    out_type="numpy"
)

h = np.abs(h_freq)[0,0,0,0,0,:]
plt.figure(figsize=(8,3))
plt.plot((freqs-28e9)/1e6, h)
plt.xlabel("Frequency offset from 28GHz [MHz]")
plt.ylabel("|H(f)|")
plt.title("CFR (non-OFDM frequency sweep)")
plt.grid(True)
plt.show()

TAP 시각화

In [ ]:
# 한글 주석: taps 설정값
bandwidth = 500e6          # 채널 대역폭 [Hz]
l_min, l_max = 0, 1500     # 탭 인덱스 범위
sampling_frequency = 10e4         # 시간축 샘플링 주파수 [Hz]
num_time_steps = 16       # 시간 샘플 수(도플러 반영)

In [ ]:
# path_loss.ipynb - taps 위젯 셀(기존 cell 28) 교체 예시
widget_view = js.create_taps_pdp_widgets(
    scene=scene,
    rx=rx,
    solver=solver,
    path_data=path_data,
    tx_names=tx_names,
    bandwidth=bandwidth,
    l_min=l_min,
    l_max=l_max,
    sampling_frequency=sampling_frequency,
    num_time_steps=num_time_steps,
    vehicle=vehicle,
    vehicle_z=0.3,
    rx_z=3.0,
)
display(widget_view)

영상 저장

In [ ]:
# 구도 확인
cam = Camera(position=[-200, -450, 150], look_at=[0, 0, 0])
scene.render(camera=cam)

In [ ]:
# path_loss.ipynb - 영상 저장 셀(기존 cell 31) 교체 예시
js.export_simulation_video(
    scene=scene,
    rx=rx,
    solver=solver,
    path_data=path_data,
    tx_names=tx_names,
    camera=cam,
    vehicle=vehicle,
    vehicle_z=0.3,
    rx_z=3.0,
    filename="HSV.mp4",
    fps=10,
    resolution=(800, 600)
)